# EDA profundo: Customer Churn Records

**Objetivo:** entender qué variables se asocian con la fuga de clientes (`Exited`), detectar problemas de calidad de datos y señalar riesgos (fuga de información, desbalance, relaciones estructurales) antes de modelar.

**Dataset:** `actividades/data/Customer-Churn-Records.csv`: 10 000 clientes de un banco.

| Columna | Descripción |
|---|---|
| `RowNumber`, `CustomerId`, `Surname` | Identificadores. No deben usarse como variables predictoras. |
| `CreditScore` | Puntuación crediticia |
| `Geography` | País del cliente |
| `Gender` | Género |
| `Age` | Edad |
| `Tenure` | Años como cliente |
| `Balance` | Saldo de la cuenta |
| `NumOfProducts` | Número de productos contratados |
| `HasCrCard` | Tiene tarjeta de crédito (0/1) |
| `IsActiveMember` | Cliente activo (0/1) |
| `EstimatedSalary` | Salario estimado |
| **`Exited`** | **Objetivo.** 1 = el cliente se fue (churn), 0 = se queda |
| `Complain` | Ha presentado una queja (0/1) |
| `Satisfaction Score` | Puntuación de satisfacción |
| `Card Type` | Tipo de tarjeta |
| `Point Earned` | Puntos acumulados |

**Índice**
1. Carga y calidad de los datos
2. Variable objetivo
3. Variables numéricas (univariante y por clase)
4. Variables categóricas y discretas (tasa de churn)
5. Asociación estadística con `Exited`
6. Análisis en profundidad (segmentos y cruces)
7. Análisis multivariante y correlaciones
8. Relaciones estructurales y riesgo de fuga de información
9. Importancia de variables (información mutua)
10. Hallazgos y próximos pasos

## 0. Configuración

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.3f}".format)
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

COLOR_SE_QUEDA = "#4C72B0"
COLOR_SE_VA = "#DD8452"

## 1. Carga y calidad de los datos

In [ ]:
RUTA = Path("../../data/Customer-Churn-Records.csv")
assert RUTA.exists(), f"No encuentro el dataset en {RUTA.resolve()}"

# Se renombran las columnas con espacios para poder usarlas como atributos y en fórmulas.
df = pd.read_csv(RUTA).rename(columns={
    "Satisfaction Score": "SatisfactionScore",
    "Card Type": "CardType",
    "Point Earned": "PointsEarned",
})

print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
df.head()

In [ ]:
ID_COLS = ["RowNumber", "CustomerId", "Surname"]
NUM_COLS = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary", "PointsEarned"]
CAT_COLS = ["Geography", "Gender", "CardType"]
BIN_COLS = ["HasCrCard", "IsActiveMember", "Complain"]
ORD_COLS = ["NumOfProducts", "SatisfactionScore"]
TARGET = "Exited"
FEATURES = NUM_COLS + CAT_COLS + BIN_COLS + ORD_COLS

In [ ]:
calidad = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "nulos": df.isna().sum(),
    "valores_unicos": df.nunique(),
    "ejemplo": df.iloc[0],
})
calidad

In [ ]:
print("Filas duplicadas (todas las columnas):", df.duplicated().sum())
print("CustomerId duplicados:", df["CustomerId"].duplicated().sum())
print("Surname duplicados:", df["Surname"].duplicated().sum(), "(normal: los apellidos se repiten)")
print()
print("Valores posibles de las variables categóricas, binarias y ordinales:")
for c in CAT_COLS + BIN_COLS + ORD_COLS:
    print(f"  {c}: {sorted(df[c].unique().tolist())}")

## 2. Variable objetivo

In [ ]:
tasa_global = df[TARGET].mean()
print(f"Tasa de churn global: {tasa_global:.2%}  ({df[TARGET].sum():,} de {len(df):,} clientes)")

conteo = df[TARGET].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.bar(["Se queda (0)", "Se va (1)"], conteo.values, color=[COLOR_SE_QUEDA, COLOR_SE_VA])
for i, v in enumerate(conteo.values):
    ax.text(i, v + 80, f"{v:,}", ha="center")
ax.set_title("Distribución de la variable objetivo")
plt.show()

## 3. Variables numéricas

### 3.1 Estadísticos descriptivos, asimetría y ceros

In [ ]:
desc = df[NUM_COLS].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).T
desc["asimetria"] = df[NUM_COLS].skew()
desc["% ceros"] = (df[NUM_COLS] == 0).mean() * 100
desc

### 3.2 Outliers (regla del rango intercuartílico, 1.5 × IQR)

In [ ]:
def contar_outliers_iqr(s: pd.Series) -> int:
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    return int(((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum())

outliers = pd.Series({c: contar_outliers_iqr(df[c]) for c in NUM_COLS}, name="outliers_IQR")
outliers.to_frame()

### 3.3 Distribución de cada variable

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), NUM_COLS):
    ax.hist(df[col], bins=40, color=COLOR_SE_QUEDA, edgecolor="white")
    ax.set_title(col)
fig.suptitle("Distribución de las variables numéricas", y=1.01)
fig.tight_layout()
plt.show()

### 3.4 Distribución por clase (`Exited`)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, col in zip(axes.ravel(), NUM_COLS):
    for valor, color, etiqueta in [(0, COLOR_SE_QUEDA, "Se queda"), (1, COLOR_SE_VA, "Se va")]:
        ax.hist(df.loc[df[TARGET] == valor, col], bins=40, density=True,
                alpha=0.5, color=color, label=etiqueta)
    ax.set_title(col)
axes[0, 0].legend()
fig.suptitle("Distribución por clase (densidad normalizada)", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
filas = []
for col in NUM_COLS:
    a = df.loc[df[TARGET] == 0, col]
    b = df.loc[df[TARGET] == 1, col]
    _, p = stats.mannwhitneyu(a, b)
    filas.append({
        "variable": col,
        "mediana_se_queda": a.median(),
        "mediana_se_va": b.median(),
        "media_se_queda": a.mean(),
        "media_se_va": b.mean(),
        "p_valor (Mann-Whitney)": f"{p:.2e}",
    })
comparacion_num = pd.DataFrame(filas).set_index("variable")
comparacion_num

El test de Mann-Whitney no asume normalidad, así que es el adecuado para variables asimétricas como `Balance` o `Age`.
Con 10 000 filas, casi cualquier diferencia sale significativa, así que el tamaño del efecto (diferencia de medianas) pesa más que el p-valor.

## 4. Variables categóricas y discretas

In [ ]:
def tabla_churn(col: str) -> pd.DataFrame:
    t = df.groupby(col, observed=True)[TARGET].agg(clientes="size", churn="sum", tasa_churn="mean")
    t["tasa_churn_%"] = t["tasa_churn"] * 100
    t["lift_vs_global"] = t["tasa_churn"] / tasa_global
    return t.drop(columns="tasa_churn")


def graficar_categorica(cols, ncols=2):
    filas_n = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(filas_n, ncols, figsize=(6 * ncols, 3.8 * filas_n))
    for ax, col in zip(axes.ravel(), cols):
        t = tabla_churn(col)
        x = t.index.astype(str)
        ax.bar(x, t["clientes"], color=COLOR_SE_QUEDA, alpha=0.6)
        ax.set_ylabel("Clientes")
        ax.set_title(col)
        ax2 = ax.twinx()
        ax2.plot(x, t["tasa_churn_%"], color=COLOR_SE_VA, marker="o", lw=2)
        ax2.axhline(tasa_global * 100, color="gray", ls="--", lw=1)
        ax2.set_ylabel("% churn")
        ax2.grid(False)
    for ax in axes.ravel()[len(cols):]:
        ax.set_visible(False)
    fig.tight_layout()
    plt.show()


graficar_categorica(["Geography", "Gender", "CardType", "NumOfProducts"])

In [ ]:
graficar_categorica(["HasCrCard", "IsActiveMember", "Complain", "SatisfactionScore"])

Barras: número de clientes. Línea naranja: % de churn en cada grupo. Línea gris discontinua: churn global.

In [ ]:
for col in CAT_COLS + BIN_COLS + ORD_COLS:
    print(f"--- {col} ---")
    display(tabla_churn(col))

## 5. Asociación estadística con `Exited`

Para variables categóricas usamos el test chi-cuadrado y la **V de Cramér** (0 = sin asociación, 1 = asociación perfecta).
Para variables numéricas, la asociación se mide con correlación en la sección 7.

In [ ]:
def cramers_v(col: str):
    tabla = pd.crosstab(df[col], df[TARGET])
    chi2, p, _, _ = stats.chi2_contingency(tabla)
    n = tabla.values.sum()
    r, k = tabla.shape
    v = np.sqrt(chi2 / (n * (min(r, k) - 1)))
    return v, p


resultados = {c: cramers_v(c) for c in CAT_COLS + BIN_COLS + ORD_COLS}
asociacion = pd.DataFrame(resultados, index=["cramers_v", "p_valor"]).T
asociacion["p_valor"] = asociacion["p_valor"].map(lambda p: f"{p:.2e}")
asociacion.sort_values("cramers_v", ascending=False)

## 6. Análisis en profundidad

### 6.1 Segmentos por edad, saldo, score crediticio y salario

In [ ]:
df["AgeGrupo"] = pd.cut(df["Age"], bins=[0, 29, 39, 49, 59, 120],
                        labels=["<30", "30-39", "40-49", "50-59", "60+"])
df["SaldoCero"] = np.where(df["Balance"] == 0, "Saldo = 0", "Saldo > 0")
df["CreditScoreGrupo"] = pd.cut(df["CreditScore"], bins=[0, 500, 600, 700, 800, 1000],
                                labels=["<500", "500-599", "600-699", "700-799", "800+"])
df["SalarioDecil"] = pd.qcut(df["EstimatedSalary"], 10, labels=range(1, 11)).astype(int)

for col in ["AgeGrupo", "SaldoCero", "CreditScoreGrupo", "SalarioDecil"]:
    print(f"--- {col} ---")
    display(tabla_churn(col))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
t = tabla_churn("Tenure")
ax.plot(t.index, t["tasa_churn_%"], marker="o", color=COLOR_SE_VA, lw=2)
ax.axhline(tasa_global * 100, color="gray", ls="--", lw=1, label="Churn global")
ax.set_xlabel("Tenure (años como cliente)")
ax.set_ylabel("% churn")
ax.set_title("Tasa de churn según antigüedad")
ax.legend()
plt.show()

### 6.2 Mapas de calor de cruces entre variables

In [ ]:
def mapa_churn(filas, columnas, titulo):
    pivot = df.pivot_table(index=filas, columns=columnas, values=TARGET, aggfunc="mean", observed=True) * 100
    conteo = df.pivot_table(index=filas, columns=columnas, values=TARGET, aggfunc="size", observed=True)

    fig, ax = plt.subplots(figsize=(1.6 * pivot.shape[1] + 3, 0.7 * pivot.shape[0] + 2))
    im = ax.imshow(pivot.values, cmap="Reds", aspect="auto", vmin=0, vmax=np.nanmax(pivot.values))
    ax.set_xticks(range(pivot.shape[1]))
    ax.set_xticklabels(pivot.columns.astype(str))
    ax.set_yticks(range(pivot.shape[0]))
    ax.set_yticklabels(pivot.index.astype(str))
    ax.set_xlabel(columnas)
    ax.set_ylabel(filas)
    ax.grid(False)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v, n = pivot.values[i, j], conteo.values[i, j]
            if np.isnan(v):
                continue
            color = "white" if v > np.nanmax(pivot.values) * 0.6 else "black"
            ax.text(j, i, f"{v:.1f}%\n(n={int(n)})", ha="center", va="center", fontsize=9, color=color)
    fig.colorbar(im, ax=ax, label="% churn", shrink=0.8)
    ax.set_title(titulo)
    fig.tight_layout()
    plt.show()


mapa_churn("Geography", "Gender", "Churn (%) por país y género")
mapa_churn("NumOfProducts", "IsActiveMember", "Churn (%) por nº de productos y actividad")
mapa_churn("Complain", "SatisfactionScore", "Churn (%) por queja y satisfacción")
mapa_churn("AgeGrupo", "Geography", "Churn (%) por edad y país")

### 6.3 Churn por país con saldo

In [ ]:
resumen_pais = df.groupby("Geography").agg(
    clientes=(TARGET, "size"),
    tasa_churn=(TARGET, "mean"),
    saldo_medio=("Balance", "mean"),
    pct_saldo_cero=("SaldoCero", lambda s: (s == "Saldo = 0").mean()),
)
resumen_pais["tasa_churn"] = resumen_pais["tasa_churn"] * 100
resumen_pais["pct_saldo_cero"] = resumen_pais["pct_saldo_cero"] * 100
resumen_pais

## 7. Análisis multivariante y correlaciones

In [ ]:
muestra = df.sample(3000, random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for ax, (x, y) in zip(axes, [("Age", "Balance"), ("CreditScore", "EstimatedSalary")]):
    for valor, color, etiqueta in [(0, COLOR_SE_QUEDA, "Se queda"), (1, COLOR_SE_VA, "Se va")]:
        s = muestra[muestra[TARGET] == valor]
        ax.scatter(s[x], s[y], s=10, alpha=0.4, color=color, label=etiqueta)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(f"{x} vs {y} (muestra de 3 000 clientes)")
axes[0].legend()
fig.tight_layout()
plt.show()

In [ ]:
corr_cols = NUM_COLS + ORD_COLS + BIN_COLS + [TARGET]
corr = df[corr_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right")
ax.set_yticks(range(len(corr_cols)))
ax.set_yticklabels(corr_cols)
ax.grid(False)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlación de Spearman (incluye la variable objetivo)")
fig.tight_layout()
plt.show()

In [ ]:
corr_con_target = corr[TARGET].drop(TARGET).sort_values(key=abs, ascending=False)
corr_con_target.to_frame("spearman_con_Exited")

## 8. Relaciones estructurales y riesgo de fuga de información

Algunas variables pueden haberse registrado **después** de que el cliente decidiera irse (por ejemplo, una queja o la puntuación de satisfacción de la salida).
Si es así, usarlas en el modelo daría métricas irreales. Aquí comprobamos cuánto explican y cómo se relacionan entre sí.

In [ ]:
print("Churn según queja:")
display(pd.crosstab(df["Complain"], df[TARGET], normalize="index").rename(columns={0: "se_queda", 1: "se_va"}))

print("Puntos ganados por tipo de tarjeta y satisfacción (mediana):")
display(df.pivot_table(index="CardType", columns="SatisfactionScore",
                       values="PointsEarned", aggfunc="median"))

print("Relación entre queja y satisfacción:")
display(pd.crosstab(df["Complain"], df["SatisfactionScore"]))

## 9. Importancia de variables (información mutua)

La información mutua captura relaciones **no lineales** entre cada variable y el objetivo, a diferencia de la correlación de Pearson/Spearman.
Las categóricas se codifican con códigos enteros solo a efectos de este cálculo.

In [ ]:
X = df[FEATURES].copy()
for c in CAT_COLS:
    X[c] = X[c].astype("category").cat.codes

es_discreta = [c in CAT_COLS + BIN_COLS + ORD_COLS for c in FEATURES]
mi = mutual_info_classif(X, df[TARGET], discrete_features=es_discreta, random_state=42)
info_mutua = pd.Series(mi, index=FEATURES, name="info_mutua").sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(info_mutua.index, info_mutua.values, color=COLOR_SE_QUEDA)
ax.set_xlabel("Información mutua con Exited")
ax.set_title("Importancia de variables (sin modelo)")
plt.show()

info_mutua.sort_values(ascending=False).to_frame()

## 10. Hallazgos y próximos pasos

Resultados principales. Las cifras proceden de las celdas anteriores.

### Calidad de los datos
- 10 000 filas y 18 columnas. Sin nulos, sin filas duplicadas y `CustomerId` único.
- Los apellidos se repiten (normal). `RowNumber`, `CustomerId` y `Surname` son identificadores y no deben entrar en el modelo.
- Hay 359 outliers en `Age`, todos en la cola alta (hasta 92 años). Son plausibles, así que no los eliminaría.

### Variable objetivo
- Churn global del **20.4 %** (2 038 de 10 000). Desbalance moderado: conviene evaluar con AUC-PR, F1 o recall, y no con accuracy.

### Riesgo principal: `Complain` (posible fuga de información)
- Con queja, el churn es del **99.5 %**; sin queja, del **0.05 %**. La V de Cramér es 0.995 y la correlación de Spearman 0.996.
- Ninguna otra variable se acerca. Es casi una copia de la etiqueta.
- Lo más probable es que la queja se registre durante el proceso de salida. Si es así, no estaría disponible al predecir, y un modelo que la use dará métricas irreales.
- `SatisfactionScore` no se relaciona con el churn (entre 19.6 % y 21.8 % en los cinco niveles) ni con las quejas. Conviene preguntar al equipo de datos qué mide realmente esa puntuación.

### Señales reales (sin `Complain`)
- **`NumOfProducts`** es la más fuerte y no es monótona: 1 producto 27.7 %, 2 productos 7.6 %, 3 productos 82.7 %, 4 productos 100 % (n = 60). Los clientes con 3 o más productos son solo el 3.3 % del total, pero casi todos se van. Por eso el Spearman (-0.125) subestima la relación.
- **`Age`** sube hasta el grupo 50-59 (56.0 % de churn) y baja a 27.9 % en 60+. Es la variable continua con más señal (Spearman 0.32; mediana 36 frente a 45).
- **`Geography`**: Germania tiene un 32.4 % de churn, frente a 16.2 % en Francia y 16.7 % en España. Además, en Germania ningún cliente tiene saldo cero (en Francia y España, cerca del 48 %) y el saldo medio es casi el doble.
- **`IsActiveMember`**: los inactivos se van un 26.9 % frente a un 14.3 % de los activos.
- **`Gender`**: churn del 25.1 % en mujeres frente al 16.5 % en hombres. Es una variable sensible: conviene revisar su uso antes de modelar.
- **`Balance`**: el 36 % de los clientes tiene saldo cero. Los de saldo positivo tienen más churn (24.1 % frente a 13.8 %), pero este efecto puede venir de Germania, porque mezcla país y saldo.

### Variables sin señal aparente
`CreditScore`, `EstimatedSalary`, `Tenure`, `HasCrCard`, `CardType`, `SatisfactionScore` y `PointsEarned` tienen V de Cramér < 0.03 o p-valores > 0.05, e información mutua ≈ 0. Conviene confirmarlo con el modelo antes de descartarlas.

### Próximos pasos
1. **Confirmar con datos o negocio cuándo se registra `Complain`.** Si es posterior a la salida, excluirla del modelo. Como referencia, entrenar también sin ella para ver la señal real.
2. **Comprobar si el efecto de Germania sigue existiendo al controlar por saldo y edad.** Así se separa el efecto del país del efecto del saldo.
3. **Ingeniería de variables:** tramos de edad (la relación no es lineal), `NumOfProducts` como categórica o con un flag para 3 o más productos, flag de país Germania y flag de saldo cero.
4. **Excluir identificadores** (`RowNumber`, `CustomerId`, `Surname`).
5. **Partición y métricas:** estratificar por `Exited` (ya está configurado con `random_state` 42) y usar AUC-PR, F1 y `class_weight` por el desbalance.